# Des probabilités à Chatgpt

## Bigram
Objectif -> prédire la prochaine lettre à partir des la précédante.
On vas donc créer un tableau d'occurence, puis le convertir en proba.

### Obtention des données
On télécharge une liste de 32 milles prénom.

In [1]:
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
with open('names.txt', 'r', encoding='utf-8') as f:
    text = f.read()
names = text.splitlines()

--2026-04-05 18:41:05--  https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
Résolution de raw.githubusercontent.com (raw.githubusercontent.com)… 2606:50c0:8002::154, 2606:50c0:8000::154, 2606:50c0:8003::154, ...
Connexion à raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443… connecté.
requête HTTP transmise, en attente de la réponse… 200 OK
Taille : 228145 (223K) [text/plain]
Enregistre : ‘names.txt’

names.txt           100%[===================>] 222,80K  --.-KB/s    ds 0,03s   

2026-04-05 18:41:06 (8,43 MB/s) - ‘names.txt’ enregistré [228145/228145]



Pour gérer des grand tableau efficacement, on utilise torch.

In [2]:
import torch

/home/arthur/Documents/code/grand-oral/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
# on prend la liste de tout les charactere présent
chars = sorted(list(set(''.join(names)))) + ["/", "|"] # slash pour le début d'un prénom, horizontale pour la fin
# on va créer un dictionnaire d'encodage
encode = {c:i for i, c in enumerate(chars)}
# et de decodage
decode = {i:c for c, i in encode.items()}

In [10]:
encode["a"], encode["t"], decode[3], decode[14]

(0, 19, 'd', 'o')

In [11]:
decode[encode["a"]]

'a'

Pour stoké les occurence on va donc faire un tableau a double entrée (une liste de listes). Le tableau d'occurence est de taille (n_lettre, n_lettre). n_lettre -> len(chars). On va donc pouvoir acceder a l'occurence d'une lettre en fonction d'une autre: tableau[encode[premiere lettre], encode[deuxieme lettre]] 

In [12]:
n_lettre = len(chars)
tableau = torch.zeros((n_lettre, n_lettre))

On va maintenant le remplir.

In [ ]:
for name in names:
    name = "/" + name + "|"
    for l1, l2 in zip(name, name[1:]):
        idxl1, idxl2 = encode[l1], encode[l2]
        tableau[idxl1, idxl2] += 1

In [ ]:
import random
def generate(lengh=5, start=None):
    start = start if start is not None else decode[random.randint(0, len(decode)-3)]
    name = "/" + start
    print(name)
    while name[-1] != "|":
        row = tableau[encode[name[-1]]]
        # temperature
        T = 1.0
        row = row / row.std(dim=0, keepdim=True) / T
        x = row - max(row)
        softmax = torch.exp(x) / torch.exp(x).sum().item()
        # top p
        threshold = max(softmax)-0.4#*max(softmax)
        proba = torch.where(softmax > threshold, softmax, 0)
        choices = [(idx, value.item()) for idx, value in enumerate(proba) if value != 0]
        nxtt = random.choice(choices)
        
        nxt = decode[nxtt[0]]
        if nxt == "|" and len(name) < lengh:
            if len(choices) > 1:
                choices.remove(nxtt)
                nxt = decode[random.choice(choices)[0]]
            else:
                topk = torch.topk(row, 3).indices.tolist()
                topk.remove(encode["|"])
                nxt = decode[topk[0]]
        name += nxt
        if len(name) > lengh+2:
            break
    return name[1:-1]

## Suite
N-gram: On veut maintennt predire la prochaine lettre en fonction des n derniere. On va donc créer un nouveau tableau de taille (len(encode)**n_lettre, len(encode)).
On peut voir que le tableau va vite devenir enorme, pour resoudre ce probleme on peut grouper des lettre en token -> tokenizer Bytes Pairs Encoding algorytme.
#### Parallele avec chatgpt
LLM = tokens -> f(tokens, parametres) -> ligne du tableau correspondant au probabilité des prochain tokens en fonction des n dernier
On va donc chercher a tuner les parametres pour avoir une fonction qui va retourner les meilleur proba.